# SingVox Phase 2D.2 — Zero-Budget Inference Evaluation
**Target Model:** `mamounyosef/sign-language-bridge` (RSLoRA on `Qwen/Qwen3-VL-2B-Instruct`)
**Compute Budget:** $0.00 (Google Colab Free Tier with T4 GPU)
**Purpose:** Run a 5-clip continuous ASL evaluation with input-dependence diagnostics (original vs. shuffled frames).

In [ ]:
# Step 1: Check GPU availability (Requires free T4 GPU backend: Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers==4.49.0 peft==0.18.1 accelerate qwen-vl-utils opencv-python mediapipe sacrebleu

In [ ]:
# Step 3: Define 5 evaluation clips and reference captions
CLIPS = [
    {
        "filename": "-d5dN54tH2E_0-1-rgb_front.mp4",
        "video_id": "-d5dN54tH2E",
        "duration_sec": 7.88,
        "reference": "We're going to work on a arm drill that will help you have graceful hand movements in front of you."
    },
    {
        "filename": "-f1_kdl050s_0-1-rgb_front.mp4",
        "video_id": "-f1_kdl050s",
        "duration_sec": 5.0,
        "reference": "In this clip we are going to talk about dangers for these birds in the household and otherwise."
    },
    {
        "filename": "00dWJ4YRRSI_0_1-1-rgb_front.mp4",
        "video_id": "00dWJ4YRRSI",
        "duration_sec": 26.58,
        "reference": "I hope this becomes very popular, here of late, is the circle hooks. Now, circle hooks has been used by Japanese fishermen and longliners for years and years, but in recent years, they've developed lighter, smaller, in shore type hooks, which are very well suited, particularly to folks that are in to the hook and release type fishing."
    },
    {
        "filename": "0oGfy530AuI_0-1-rgb_front.mp4",
        "video_id": "0oGfy530AuI",
        "duration_sec": 1.29,
        "reference": "MICHAEL ROSENGART: Thrusters."
    },
    {
        "filename": "0pKzG0RRUz4_0-1-rgb_front.mp4",
        "video_id": "0pKzG0RRUz4",
        "duration_sec": 13.08,
        "reference": "Another technique is you can braid the hair on first and then start wrapping cause some people have really short hair and we can create the look with adding the hair end."
    }
]
print(f"Configured {len(CLIPS)} evaluation clips from distinct source videos.")

In [ ]:
# Step 4: Load Base Model and RSLoRA Adapter
import torch, time
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
ADAPTER_ID = "mamounyosef/sign-language-bridge"
ADAPTER_SUBDIR = "adapter"

print("Loading base model in bfloat16 on GPU...")
t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

print("Loading RSLoRA adapter...")
model = PeftModel.from_pretrained(
    model,
    ADAPTER_ID,
    subfolder=ADAPTER_SUBDIR,
    is_trainable=False
)
model = model.merge_and_unload()
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s. VRAM allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")

In [ ]:
# Step 5: Define Inference and Shuffling Utilities
import cv2, random, numpy as np
from qwen_vl_utils import process_vision_info

def run_video_inference(model, processor, video_path):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a sign language translator."}]},
        {"role": "user", "content": [
            {"type": "video", "video": video_path, "fps": 2, "min_pixels": 4*32*32, "max_pixels": 256*256},
            {"type": "text", "text": "Translate this American Sign Language video into English."}
        ]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos, video_kwargs = process_vision_info(messages, return_video_kwargs=True)
    inputs = processor(text=[text], images=images if images else None, videos=videos if videos else None, return_tensors="pt", **video_kwargs)
    inputs = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    
    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            num_beams=5,
            length_penalty=0.6,
            no_repeat_ngram_size=4,
            repetition_penalty=1.1,
            max_new_tokens=32
        )
    lat_ms = (time.time() - t0) * 1000.0
    gen_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    hyp = processor.tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return hyp, lat_ms

print("Inference pipeline initialized successfully.")